<a href="https://colab.research.google.com/github/xhoja/Deep-LearningHeritagePreservation/blob/main/notebooks/03_segmentation_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Crack Segmentation — MAnet + mit_b2
Colab version. GPU-intensive 3-phase training pipeline.

**Setup**:
1. Open in Colab (or new Colab notebook)
2. Runtime → Change runtime type → T4 GPU
3. Run all cells

Cell 3 clones repo from GitHub and extracts datasets automatically.

**Output**: Best checkpoint + training curves to `/content/checkpoints` (auto-downloadable)

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU — Runtime > Change runtime type > T4 GPU"
print(f"GPU   : {torch.cuda.get_device_name(0)}")
print(f"VRAM  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"PyTorch: {torch.__version__}")

# Mount Google Drive for checkpoint backup
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print("Google Drive mounted for backup.")

In [ ]:
!pip install -q segmentation-models-pytorch albumentations==1.4.3 timm

In [ ]:
from pathlib import Path
import json
import zipfile
import shutil

# Clone repo & extract datasets
print("Cloning repo...")
import subprocess
subprocess.run(['git', 'clone', 'https://github.com/xhoja/Deep-LearningHeritagePreservation.git',
                '/content/repo'], check=True)

WORK_DIR  = Path('/content')
DATA_DIR  = WORK_DIR / 'data'
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Extract all zips from repo/data
repo_data = Path('/content/repo/data')
print("Extracting datasets...")

for zip_file in repo_data.glob('*.zip'):
    print(f"  {zip_file.name}...")
    with zipfile.ZipFile(zip_file, 'r') as z:
        z.extractall(DATA_DIR)

for subdir in repo_data.glob('*/'):
    if subdir.is_dir():
        for zip_file in subdir.glob('*.zip'):
            print(f"  {zip_file.name}...")
            with zipfile.ZipFile(zip_file, 'r') as z:
                z.extractall(DATA_DIR / subdir.name)

CKPT_DIR  = WORK_DIR / 'checkpoints'
PLOTS_DIR = WORK_DIR / 'plots'

CKPT_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

# Setup Drive backup
DRIVE_CKPT = Path('/content/drive/MyDrive/segmentor_v4_backup')
DRIVE_CKPT.mkdir(parents=True, exist_ok=True)

def backup_to_drive():
    """Save checkpoint & history to Google Drive."""
    if (CKPT_DIR / 'best.pth').exists():
        shutil.copy2(CKPT_DIR / 'best.pth', DRIVE_CKPT / 'best.pth')
    if (CKPT_DIR / 'history.json').exists():
        shutil.copy2(CKPT_DIR / 'history.json', DRIVE_CKPT / 'history.json')
    print(f"  Backed up to Drive: {DRIVE_CKPT}")

print(f"Data   : {DATA_DIR}")
print(f"Working: {WORK_DIR}")
print(f"Drive backup: {DRIVE_CKPT}")

In [ ]:
import os
print('Extracted datasets:')
if DATA_DIR.exists():
    for item in sorted(os.listdir(DATA_DIR)):
        p = DATA_DIR / item
        count = sum(1 for _ in p.rglob('*') if _.is_file()) if p.is_dir() else 0
        print(f'  {item}/  ({count} files)' if p.is_dir() else f'  {item}')
else:
    print(f"  DATA_DIR not found: {DATA_DIR}")

In [ ]:
import cv2
import numpy as np
import random
from collections import Counter
from sklearn.model_selection import train_test_split

IMG_EXTS       = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}
IMG_DIR_NAMES  = {'images', 'img', 'image', 'jpegimages', 'rgb', 'data'}
MASK_DIR_NAMES = {'masks', 'mask', 'labels', 'annotations', 'gt', 'ground_truth', 'seg'}

def find_dir(root, dir_names):
    """Find a directory matching any name in dir_names."""
    root_p = Path(root)
    for d in root_p.rglob('*'):
        if d.is_dir() and d.name.lower() in dir_names:
            return d
    return None

def find_pairs(root):
    """Find all (image, mask) pairs in nested directories."""
    root_p = Path(root)
    img_dir = find_dir(root, IMG_DIR_NAMES) or root_p
    mask_dir = find_dir(root, MASK_DIR_NAMES) or root_p

    pairs = []
    for img in sorted(img_dir.rglob('*')):
        if img.is_file() and img.suffix.lower() in IMG_EXTS:
            stem = img.stem
            mask = mask_dir / f"{stem}.png"
            if not mask.exists():
                mask = mask_dir / f"{stem}.jpg"
            if mask.exists():
                pairs.append((img, mask))
    return pairs

# Load all datasets
all_pairs = []
for dataset_dir in sorted(DATA_DIR.iterdir()):
    if dataset_dir.is_dir():
        pairs = find_pairs(dataset_dir)
        all_pairs.extend(pairs)
        print(f"{dataset_dir.name:30s} {len(pairs):4d} pairs")

print(f"\nTotal: {len(all_pairs)} pairs")

# Split
random.seed(42)
train_p, temp_p = train_test_split(all_pairs, train_size=0.7, random_state=42)
val_p, test_p = train_test_split(temp_p, train_size=0.5, random_state=42)

print(f"Train: {len(train_p)} | Val: {len(val_p)} | Test: {len(test_p)}")

In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

def get_seg_transforms(split, img_size=384):
    if split == 'train':
        return A.Compose([
            A.Resize(img_size, img_size),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.3),
            A.RandomBrightnessContrast(p=0.3),
            A.GaussNoise(p=0.1),
            A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
            ToTensorV2(),
        ])
    else:
        return A.Compose([
            A.Resize(img_size, img_size),
            A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
            ToTensorV2(),
        ])

In [ ]:
from torch.utils.data import Dataset, DataLoader

_CLAHE = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))

class CrackSegDataset(Dataset):
    def __init__(self, pairs, split='train', img_size=384, transform=None):
        self.pairs     = pairs
        self.split     = split
        self.img_size  = img_size
        self.transform = transform

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_path, mask_path = self.pairs[idx]
        img = cv2.imread(str(img_path), cv2.IMREAD_COLOR)
        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)

        if img is None or mask is None:
            return self[random.randint(0, len(self) - 1)]

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = _CLAHE.apply(cv2.cvtColor(img, cv2.COLOR_RGB2GRAY))
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)

        if self.transform:
            aug = self.transform(image=img, mask=mask)
            img, mask = aug['image'], aug['mask']

        mask = (mask > 127).astype(np.float32)
        mask = torch.from_numpy(mask).unsqueeze(0)
        return img, mask

def make_loaders(train_pairs, val_pairs, test_pairs, img_size, batch_size, batch_eval=4):
    train_ds = CrackSegDataset(train_pairs, split='train', img_size=img_size,
                               transform=get_seg_transforms('train', img_size))
    val_ds   = CrackSegDataset(val_pairs,   split='val',   img_size=img_size,
                               transform=get_seg_transforms('val', img_size))
    test_ds  = CrackSegDataset(test_pairs,  split='test',  img_size=img_size,
                               transform=get_seg_transforms('val', img_size))

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader   = DataLoader(val_ds,   batch_size=batch_eval, shuffle=False, num_workers=2)
    test_loader  = DataLoader(test_ds,  batch_size=batch_eval, shuffle=False, num_workers=2)

    return train_loader, val_loader, test_loader, test_ds

In [ ]:
DEVICE = torch.device('cuda')

In [ ]:
import segmentation_models_pytorch as smp
import torch.nn as nn

ENCODER = 'mit_b2'

def build_model(encoder=ENCODER):
    return smp.MAnet(
        encoder_name=encoder,
        encoder_weights='imagenet',
        in_channels=3,
        classes=1,
        activation=None,
    )

model = build_model().to(DEVICE)
print(f"Model: MAnet + {ENCODER}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

In [ ]:
tversky_loss = smp.losses.TverskyLoss(mode='binary', alpha=0.3, beta=0.7, from_logits=True)
lovasz_loss  = smp.losses.LovaszLoss(mode='binary', per_image=False, from_logits=True)

def combined_loss(pred, target):
    return 0.5 * tversky_loss(pred, target) + 0.5 * lovasz_loss(pred, target)

print(f"Loss: 0.5*Tversky(alpha=0.3,beta=0.7) + 0.5*Lovasz")

In [ ]:
from tqdm.notebook import tqdm
from torch.amp import GradScaler, autocast

def compute_seg_metrics(pred_logits, target, threshold=0.5):
    pred_binary = (torch.sigmoid(pred_logits) > threshold).long()
    tp, fp, fn, tn = smp.metrics.get_stats(pred_binary, target.long(), mode='binary')
    return {
        'iou': smp.metrics.iou_score(tp, fp, fn, tn, reduction='micro').item(),
        'dice': smp.metrics.f1_score(tp, fp, fn, tn, reduction='micro').item(),
    }

def run_epoch(model, loader, optimizer, scaler, train=True):
    model.train() if train else model.eval()
    losses, ious = [], []

    for img, mask in tqdm(loader, disable=not train):
        img, mask = img.to(DEVICE), mask.to(DEVICE)

        with autocast('cuda'):
            pred = model(img)
            loss = combined_loss(pred, mask)

        if train:
            optimizer.zero_grad()
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            with torch.no_grad():
                metrics = compute_seg_metrics(pred, mask)
                ious.append(metrics['iou'])

        losses.append(loss.item())

    return {
        'loss': np.mean(losses),
        'iou': np.mean(ious) if ious else 0,
    }

## Phase 1: Base training @ 256px

In [ ]:
PHASE1_EPOCHS = 20
PHASE1_SIZE   = 256
PHASE1_BATCH  = 32
PHASE1_LR     = 1e-3

train_loader, val_loader, test_loader, test_ds = make_loaders(
    train_p, val_p, test_p, PHASE1_SIZE, PHASE1_BATCH, batch_eval=4)

optimizer = torch.optim.AdamW(model.parameters(), lr=PHASE1_LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=PHASE1_EPOCHS, eta_min=1e-7)
scaler = GradScaler('cuda')

history = {k: [] for k in ['train_loss', 'train_iou', 'val_loss', 'val_iou', 'val_dice']}
best_iou_p1 = 0.0
torch.cuda.empty_cache()

print(f'Phase 1: img_size={PHASE1_SIZE}  epochs={PHASE1_EPOCHS}  lr={PHASE1_LR}  batch={PHASE1_BATCH}')

for epoch in range(1, PHASE1_EPOCHS + 1):
    tr = run_epoch(model, train_loader, optimizer, scaler, train=True)
    va = run_epoch(model, val_loader,   optimizer, scaler, train=False)
    scheduler.step()
    for k, v in [('train_loss', tr['loss']), ('train_iou', tr['iou']),
                   ('val_loss', va['loss']), ('val_iou', va['iou'])]:
        history[k].append(v)

    if va['iou'] > best_iou_p1:
        best_iou_p1 = va['iou']
        torch.save({'model_state': model.state_dict(), 'val_iou': best_iou_p1},
                   CKPT_DIR / 'best.pth')
        improved = ' ✓'
    else:
        improved = ''

    if epoch % 5 == 0:
        print(f'P1 E{epoch:2d}  train_loss={tr["loss"]:.4f}  val_iou={va["iou"]:.4f}{improved}')

print(f'Phase 1 done. Best val_iou={best_iou_p1:.4f}')
backup_to_drive()

## Phase 2: Refinement @ 384px

In [ ]:
PHASE2_EPOCHS = 70
PHASE2_SIZE   = 384
PHASE2_BATCH  = 16
PHASE2_LR     = 5e-4

train_loader, val_loader, test_loader, test_ds = make_loaders(
    train_p, val_p, test_p, PHASE2_SIZE, PHASE2_BATCH, batch_eval=4)

optimizer = torch.optim.AdamW(model.parameters(), lr=PHASE2_LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=PHASE2_EPOCHS, eta_min=1e-7)
scaler = GradScaler('cuda')

best_iou_p2 = best_iou_p1
torch.cuda.empty_cache()

print(f'Phase 2: img_size={PHASE2_SIZE}  epochs={PHASE2_EPOCHS}  lr={PHASE2_LR}  batch={PHASE2_BATCH}')

for epoch in range(1, PHASE2_EPOCHS + 1):
    tr = run_epoch(model, train_loader, optimizer, scaler, train=True)
    va = run_epoch(model, val_loader,   optimizer, scaler, train=False)
    scheduler.step()
    for k, v in [('train_loss', tr['loss']), ('train_iou', tr['iou']),
                   ('val_loss', va['loss']), ('val_iou', va['iou'])]:
        history[k].append(v)

    if va['iou'] > best_iou_p2:
        best_iou_p2 = va['iou']
        torch.save({'model_state': model.state_dict(), 'val_iou': best_iou_p2},
                   CKPT_DIR / 'best.pth')
        improved = ' ✓'
    else:
        improved = ''

    if epoch % 10 == 0:
        print(f'P2 E{epoch:2d}  train_loss={tr["loss"]:.4f}  val_iou={va["iou"]:.4f}{improved}')

print(f'Phase 2 done. Best val_iou={best_iou_p2:.4f}')
backup_to_drive()

## Phase 3: High-res @ 512px

In [ ]:
PHASE3_EPOCHS = 30
PHASE3_SIZE   = 512
PHASE3_BATCH  = 8
PHASE3_LR     = 3e-5

ckpt = torch.load(CKPT_DIR / 'best.pth', map_location=DEVICE)
model.load_state_dict(ckpt['model_state'])
print(f'Loaded best from phase 2  (val_iou={ckpt["val_iou"]:.4f})')

train_loader, val_loader, test_loader, test_ds = make_loaders(
    train_p, val_p, test_p, PHASE3_SIZE, PHASE3_BATCH, batch_eval=4)

optimizer = torch.optim.AdamW(model.parameters(), lr=PHASE3_LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=PHASE3_EPOCHS, eta_min=1e-7)
scaler = GradScaler('cuda')

best_iou_p3 = ckpt['val_iou']
torch.cuda.empty_cache()

print(f'Phase 3: img_size={PHASE3_SIZE}  epochs={PHASE3_EPOCHS}  lr={PHASE3_LR}  batch={PHASE3_BATCH}')

for epoch in range(1, PHASE3_EPOCHS + 1):
    tr = run_epoch(model, train_loader, optimizer, scaler, train=True)
    va = run_epoch(model, val_loader,   optimizer, scaler, train=False)
    scheduler.step()
    for k, v in [('train_loss', tr['loss']), ('train_iou', tr['iou']),
                   ('val_loss', va['loss']), ('val_iou', va['iou'])]:
        history[k].append(v)

    if va['iou'] > best_iou_p3:
        best_iou_p3 = va['iou']
        torch.save({'model_state': model.state_dict(), 'val_iou': best_iou_p3},
                   CKPT_DIR / 'best.pth')
        improved = ' ✓'
    else:
        improved = ''

    if epoch % 10 == 0:
        print(f'P3 E{epoch:2d}  train_loss={tr["loss"]:.4f}  val_iou={va["iou"]:.4f}{improved}')

print(f'Phase 3 done. Best val_iou={best_iou_p3:.4f}')
backup_to_drive()

## Evaluation on test set

In [ ]:
ckpt = torch.load(CKPT_DIR / 'best.pth', map_location=DEVICE)
model.load_state_dict(ckpt['model_state'])
model.eval()

# Test evaluation
test_iou_scores = []
test_dice_scores = []
crack_iou_scores = []

with torch.no_grad():
    for img, mask in tqdm(test_loader, desc='Test eval'):
        img, mask = img.to(DEVICE), mask.to(DEVICE)
        pred = model(img)

        pred_binary = (torch.sigmoid(pred) > 0.5).long()
        tp, fp, fn, tn = smp.metrics.get_stats(pred_binary, mask.long(), mode='binary')

        iou = smp.metrics.iou_score(tp, fp, fn, tn, reduction='micro').item()
        dice = smp.metrics.f1_score(tp, fp, fn, tn, reduction='micro').item()

        test_iou_scores.append(iou)
        test_dice_scores.append(dice)
        crack_iou_scores.append(iou)  # For crack IoU

test_results = {
    'crack_iou': np.mean(crack_iou_scores),
    'mean_iou': np.mean(test_iou_scores),
    'dice': np.mean(test_dice_scores),
}

print(f'\nTest Results:')
print(f'  Crack IoU: {test_results["crack_iou"]:.4f}')
print(f'  mIoU     : {test_results["mean_iou"]:.4f}')
print(f'  Dice     : {test_results["dice"]:.4f}')

In [ ]:
import matplotlib.pyplot as plt

total_epochs = len(history['train_loss'])
ep = range(1, total_epochs + 1)
p1_end = PHASE1_EPOCHS
p2_end = PHASE1_EPOCHS + PHASE2_EPOCHS

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle(f'MAnet + {ENCODER} — 3-Phase Training ({total_epochs} epochs total)')

for ax, (y1, y2, title, target) in zip(axes, [
    (history['train_loss'], history['val_loss'],  'Loss',     None),
    (history['train_iou'],  history['val_iou'],   'IoU',      0.90),
    (history['val_iou'],    None,                 'Val IoU (phases)', None),
]):
    ax.plot(ep, y1, label='train')
    if y2 is not None:
        ax.plot(ep, y2, label='val')
    if target is not None:
        ax.axhline(target, color='r', linestyle='--', label=f'target {target}')
    ax.axvline(p1_end, color='gray', linestyle=':', alpha=0.7, label='P1->P2')
    ax.axvline(p2_end, color='gray', linestyle='--', alpha=0.7, label='P2->P3')
    ax.set_title(title)
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(PLOTS_DIR / 'training_curves.png', dpi=150)
plt.show()

# Save history
with open(CKPT_DIR / 'history.json', 'w') as f:
    json.dump({k: [float(v) for v in vs] for k, vs in history.items()}, f)

print(f'\n=== Training Complete ===")
print(f'  Architecture : MAnet + {ENCODER}')
print(f'  Training     : {PHASE1_EPOCHS}ep@{PHASE1_SIZE}px + {PHASE2_EPOCHS}ep@{PHASE2_SIZE}px + {PHASE3_EPOCHS}ep@{PHASE3_SIZE}px')
print(f'  Loss         : Tversky(alpha=0.3,beta=0.7) + Lovasz')
print(f'  Crack IoU    : {test_results["crack_iou"]:.4f}')
print(f'  mIoU         : {test_results["mean_iou"]:.4f}')
print(f'  Dice         : {test_results["dice"]:.4f}')
print(f'  Checkpoint   : {CKPT_DIR}/best.pth')